# winnex-madhava — Strict Benchmark (GT-in-subset, vs HNSW, scaling)

**A methodology-honest follow-up to the first benchmark.**

The first notebook compared against the *official* BIGANN-100M ground truth,
which was computed over the full 100M corpus. On a 10M subset that inflates
the "efficiency" number (even a perfect scan caps at R@10≈0.43). This notebook
fixes that and answers four questions the README cannot claim yet:

| # | Question | How we measure it |
|---|---|---|
| 1 | **Scales to 100M+ with competitive latency?** | index 10M → 50M → 100M of BIGANN, measure build + latency |
| 2 | **Parity with strong approximate indexes?** | FAISS HNSW on the *same* subset, GT recomputed in-subset |
| 3 | **Robust across data distributions?** | synthetic gaussian-clustered / uniform / low-dim corpora |
| 4 | **Memory + build cost at scale?** | `resource.getrusage` maxRSS + build wall-time |

**Key methodological fix:** ground truth is recomputed *within the indexed
subset* by an exact brute-force scan. No GT misalignment, no inflated
efficiency. If `winnex-madhava` still matches the in-subset exact scan and
stays within a small margin of FAISS HNSW, the claims are real.

In [ ]:
# 1. Install winnex-madhava (published PyPI wheel) + FAISS for the baseline.
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'winnex-madhava'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'faiss-cpu'])
print('installed winnex-madhava + faiss-cpu')

In [ ]:
import json, os, time, glob, resource, gc, warnings
import numpy as np
warnings.filterwarnings('ignore')

import winnex_madhava
import faiss
print('winnex_madhava', winnex_madhava.__version__)
print('faiss', faiss.__version__)
print('CPU threads:', os.cpu_count())

## 2. In-subset ground truth helper

For a corpus and query, the true top-K is computed by a brute-force L2 scan
over *that corpus* (vectorized, numba-free). This is the ground truth both
indexes are measured against — no external GT file, no subset misalignment.

In [ ]:
# 2. In-subset ground truth: brute-force exact L2 top-K.
def exact_topk_batch(corpus_f32, queries_f32, k):
    """Return per-query sorted top-k ids by exact L2² over the corpus."""
    gt = []
    for q in queries_f32:
        diff = corpus_f32 - q
        l2 = np.einsum('ij,ij->i', diff, diff)
        gt.append(np.argsort(l2)[:k].tolist())
    return gt

def recall_at_k(ann_ids, gt_ids, k):
    a, g = set(ann_ids[:k]), set(gt_ids[:k])
    return len(a & g) / max(k, 1)

## 3. BIGANN scaling: 10M → 50M → 100M

Uses the attached `shurangwu/bigann-100m` dataset (base.u8bin, 12.8 GB). The
corpus is memory-mapped; winnex-madhava reads it without copying. We measure
build time, per-query latency, and peak RSS for each scale. Ground truth is the
in-subset exact scan for a small query set.

In [ ]:
# 3. Locate the BIGANN dataset recursively.
def find_file(name):
    for r, d, files in os.walk('/kaggle/input/'):
        if name in files:
            return os.path.join(r, name)
    return None

base_path = find_file('base.u8bin')
qpath = find_file('unif_query_10k.u8bin')
print('base:', base_path)
print('queries:', qpath)

DIM = 128
K = 10
has_bigann = base_path is not None and qpath is not None
print('BIGANN available:', has_bigann)

In [ ]:
# 4. Scaling sweep on BIGANN (if available).
import psutil

scaling_results = []

def max_rss_gb():
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6  # Linux KB->GB

if has_bigann:
    base = np.memmap(base_path, dtype=np.uint8, mode='r', shape=(100_000_000, DIM))
    # Load a small query set (first 20 real queries).
    nq = 20
    qbuf = np.fromfile(qpath, dtype=np.uint8, count=nq * 2 * DIM).reshape(-1, DIM)
    queries = qbuf[::2].astype(np.float32)  # stride 2, first 20

    for N in (10_000_000, 50_000_000, 100_000_000):
        corpus = base[:N]
        print(f'\n=== N={N:,} ===')

        # Build
        gc.collect()
        t0 = time.time()
        engine = winnex_madhava.build_engine(corpus, dim=DIM, k=K, k1_fraction=0.05, postfilter=True)
        build_s = time.time() - t0
        rss_gb = max_rss_gb()

        # Latency (warm, 20 queries)
        lat = []
        for q in queries:
            res = engine.search(q)
            lat.append(res.latency_ms)

        print(f'build: {build_s:.1f}s | lat: {np.mean(lat):.1f} ms/query | peak RSS: {rss_gb:.1f} GB')
        print(f'bound violations: {res.bound_violations}')
        scaling_results.append({
            'N': N, 'build_s': round(build_s, 2),
            'lat_ms': round(float(np.mean(lat)), 2),
            'rss_gb': round(rss_gb, 2), 'violations': int(res.bound_violations),
        })
        del engine, corpus
        gc.collect()
else:
    print('BIGANN not mounted — skipping real scaling, using synthetic proxy below.')

In [ ]:
# 5. Show the scaling table.
print('\n=== Scaling summary ===')
print(f"{'N':>12} {'build(s)':>10} {'lat(ms)':>10} {'RSS(GB)':>10} {'vio':>5}")
for r in scaling_results:
    print(f"{r['N']:>12,} {r['build_s']:>10.1f} {r['lat_ms']:>10.1f} {r['rss_gb']:>10.1f} {r['violations']:>5}")

## 4. Parity vs FAISS HNSW (in-subset GT)

On a fixed 1M subset of BIGANN (or synthetic), we compare:
- `winnex-madhava` bound+post-filter
- `faiss.IndexHNSWFlat` (M=32, efSearch=128) — a strong approximate baseline

Both are scored against the **in-subset exact scan** ground truth. We report
R@10, NDCG@10, latency, build time, and (for winnex) bound violations.

In [ ]:
# 6. Fixed-size comparison corpus: first 1M BIGANN vectors if available, else synthetic.
N_CMP = 1_000_000
if has_bigann:
    cmp_corpus = np.ascontiguousarray(base[:N_CMP], dtype=np.uint8)
    data_name = 'BIGANN-1M subset'
else:
    rng = np.random.default_rng(7)
    cmp_corpus = rng.integers(0, 256, size=(N_CMP, DIM), dtype=np.uint8)
    data_name = 'synthetic uniform 1M'

cmp_f32 = cmp_corpus.astype(np.float32)
cmp_queries = cmp_f32[:40]  # 40 in-corpus queries
print(f'comparison corpus: {data_name}, {cmp_corpus.shape}')

In [ ]:
# 7. In-subset ground truth for the comparison set.
gt_cmp = exact_topk_batch(cmp_f32, cmp_queries, K)
print('ground truth computed (in-subset exact scan)')

In [ ]:
# 8. Build winnex-madhava + FAISS HNSW on the same corpus.

# winnex-madhava
t0 = time.time()
wm = winnex_madhava.build_engine(cmp_corpus, dim=DIM, k=K, k1_fraction=0.05, postfilter=True)
wm_build = time.time() - t0

# FAISS HNSW (L2). Note: HNSW is built from float32; the uint8 corpus is
# converted once (this is the standard FAISS usage).
t0 = time.time()
hnsw = faiss.IndexHNSWFlat(DIM, 32)
hnsw.hnsw.efConstruction = 200
hnsw.hnsw.efSearch = 128
hnsw.add(cmp_f32)
hnsw_build = time.time() - t0

print(f'winnex-madhava build: {wm_build:.2f}s')
print(f'FAISS HNSW build:     {hnsw_build:.2f}s')

In [ ]:
# 9. Score both against in-subset GT: R@10, NDCG@10, latency.
def ndcg_at_k(ann_ids, gt_ids, k):
    rel = {v: 1 for v in gt_ids[:k]}
    dcg = sum((rel.get(a, 0)) / np.log2(i + 2) for i, a in enumerate(ann_ids[:k]))
    idcg = sum(1.0 / np.log2(i + 2) for i in range(k))
    return dcg / idcg if idcg else 0.0

def eval_engine(method_name, search_fn, queries, gt):
    r10s, ndcgs, lats, viol = [], [], [], 0
    for q, g in zip(queries, gt):
        t0 = time.time()
        res = search_fn(q)
        dt = (time.time() - t0) * 1000
        ann = res.indices if hasattr(res, 'indices') else res.tolist()
        r10s.append(recall_at_k(ann, g, K))
        ndcgs.append(ndcg_at_k(ann, g, K))
        lats.append(dt)
        if hasattr(res, 'bound_violations'):
            viol += res.bound_violations
    return {'method': method_name, 'R@10': np.mean(r10s), 'NDCG': np.mean(ndcgs),
            'lat_ms': np.mean(lats), 'violations': viol}

wm_res = eval_engine('winnex-madhava', wm.search, cmp_queries, gt_cmp)
hnsw_res = eval_engine('FAISS-HNSW', lambda q: hnsw.search(q.reshape(1, -1), K)[1].flatten().astype(int), cmp_queries, gt_cmp)
exact_res = eval_engine('exact_scan', wm.search_exact, cmp_queries, gt_cmp)

print(f"{'method':<16} {'R@10':>7} {'NDCG':>7} {'lat_ms':>8} {'vio':>5}")
for r in (exact_res, wm_res, hnsw_res):
    print(f"{r['method']:<16} {r['R@10']:>7.4f} {r['NDCG']:>7.4f} {r['lat_ms']:>8.2f} {r['violations']:>5}")

print(f"\nwinnex vs exact  : {100*wm_res['R@10']/exact_res['R@10']:.1f}% of ceiling")
print(f"winnex vs HNSW   : {100*wm_res['R@10']/hnsw_res['R@10']:.1f}% of HNSW R@10")

## 5. Robustness across data distributions

On small synthetic corpora (100K), we compare winnex-madhava vs exact scan
across distributions: gaussian clusters, uniform, and low-dim (16D). This
tests whether the QR-projection bound stays tight outside BIGANN-style data.

In [ ]:
# 10. Robustness sweep.
def make_corpus(dist, n=100_000, d=DIM):
    rng = np.random.default_rng(hash(dist) % 2**32)
    if dist == 'gauss_clusters':
        centers = rng.normal(0, 10, size=(16, d))
        idx = rng.integers(0, 16, size=n)
        x = centers[idx] + rng.normal(0, 2, size=(n, d))
        return np.clip(x, 0, 255).astype(np.uint8)
    if dist == 'uniform':
        return rng.integers(0, 256, size=(n, d)).astype(np.uint8)
    if dist == 'low_dim':
        x = rng.integers(0, 256, size=(n, 16)).astype(np.uint8)
        return np.tile(x, (1, d // 16))  # repeated structure
    raise ValueError(dist)

robust_results = []
for dist in ('gauss_clusters', 'uniform', 'low_dim'):
    X = make_corpus(dist)
    Xf = X.astype(np.float32)
    qs = Xf[:30]
    gt = exact_topk_batch(Xf, qs, K)

    eng = winnex_madhava.build_engine(X, dim=DIM, k=K, k1_fraction=0.02, postfilter=True)
    r10s, viol = [], 0
    for q, g in zip(qs, gt):
        res = eng.search(q)
        r10s.append(recall_at_k(res.indices, g, K))
        viol += res.bound_violations
    r10 = float(np.mean(r10s))
    robust_results.append({'dist': dist, 'R@10': round(r10, 4), 'violations': int(viol)})
    print(f'{dist:>14}: R@10={r10:.4f}  violations={viol}')
    del X, eng; gc.collect()

In [ ]:
# 11. Save all results.
report = {
    'package': 'winnex-madhava',
    'version': winnex_madhava.__version__,
    'methodology': 'in-subset exact-scan ground truth (no official-GT misalignment)',
    'dim': DIM, 'k': K,
    'bigann_dataset_available': has_bigann,
    'comparison_data': data_name,
    'scaling': scaling_results,
    'vs_hnsw': {'exact_scan': exact_res, 'winnex_madhava': wm_res, 'faiss_hnsw': hnsw_res},
    'robustness': robust_results,
}
with open('/kaggle/working/winnex_madhava_strict_results.json', 'w') as f:
    json.dump(report, f, indent=2)

import csv
with open('/kaggle/working/winnex_madhava_strict_results.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['section', 'item', 'R@10', 'NDCG', 'lat_ms', 'violations', 'build_s', 'rss_gb'])
    for r in scaling_results:
        w.writerow(['scaling', r['N'], '', '', r['lat_ms'], r['violations'], r['build_s'], r['rss_gb']])
    for sec, r in [('exact', exact_res), ('winnex', wm_res), ('hnsw', hnsw_res)]:
        w.writerow(['vs_hnsw', sec, r['R@10'], r['NDCG'], r['lat_ms'], r['violations']])
    for r in robust_results:
        w.writerow(['robust', r['dist'], r['R@10'], '', '', r['violations']])

print('\nsaved /kaggle/working/winnex_madhava_strict_results.json + .csv')
print(json.dumps(report, indent=2)[:1200])

## Summary

This benchmark is deliberately strict:
- **Ground truth is recomputed in-subset** — no inflated efficiency.
- **FAISS HNSW is the baseline** — the honest bar for a strong approximate index.
- **Scaling to 100M** — real build/latency/memory numbers.
- **Robustness across distributions** — outside BIGANN-style data.

The numbers are saved to the working directory for full transparency. If
`winnex-madhava` reaches ~100% of the in-subset exact ceiling *and* stays
competitive with FAISS HNSW on recall, the product claims are defensible. If
HNSW wins on latency, this notebook says so — that is the honest trade-off
documented in the README.